# Data Description
- Author: RUEI-HSIANG, CHANG（B12303030；AI 協助）
- Background: 四類候選人盤點的官方來源存取紀錄；新下載內政部個人頁與中選會名冊、審定入口。
- date received: 20261019
- Path to data file: 下方 manifest 的 url 欄保存逐筆官方來源。
- Unit of observation: 一筆 GET 回應；HTML 為原始 bytes，manifest 為擷取日誌。
- Sample period: 2026-10-19 擷取快照；內政部人物頁可能沿用111年選舉履歷，不等於2026更新內容。
- Known issues: 只下載與議員登記同縣市同姓名之名錄線索，非全體候選人背景；身分尚待核對。HTTP 200 不保證正文完整。中選會頁框沒有可讀審定內文。
- Definition for each variable:
  - source_id: 原內政部 official_key 或中選會入口標籤。
  - url / expected_name / county_city: 來源網址及名錄線索中的姓名、縣市。
  - captured_date / http_status / error: 擷取日、HTTP 狀態與實際錯誤；error 空白不是內容完整證明。
  - path / sha256: 保存原始回應相對路徑及 SHA-256。
  - name_present: 解析後全頁是否出現預期姓名，非獨立身分確認。
  - education_raw / experience_raw: .essay 中學歷／簡歷標籤原文；無值保留空白，不解讀為無學歷／經歷。
- Data path: `data/raw/inventory_20261019/capture_manifest.json` 及其中逐筆 path。
- Role: raw/source-access-audit
- Verification status: 下方實際 hash、JSON schema、頁面數與範例檢查；不認證人物身分與資格。
- SHA-256: `265f37a902791c15b2b615a9d02ecac4329340d42032cbfd85638d95bb6eb609`；逐頁 hash 見 manifest。
- Next processing notebook: `data/code/prepare_candidate_inventory_20261019.ipynb`


In [1]:
from pathlib import Path
import hashlib,json
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()


## Change for Your Own Data
固定來源 manifest，不重新下載或寫入原始資料。

In [2]:
MANIFEST='data/raw/inventory_20261019/capture_manifest.json'
EXPECTED='265f37a902791c15b2b615a9d02ecac4329340d42032cbfd85638d95bb6eb609'
assert sha(ROOT/MANIFEST)==EXPECTED
records=json.loads((ROOT/MANIFEST).read_text())
df=pd.DataFrame(records)

## Summary

In [3]:
print('回應筆數',len(df),'HTTP狀態',df.http_status.value_counts().to_dict())
print('學歷非空',sum(df.education_raw!=''),'簡歷非空',sum(df.experience_raw!=''))

回應筆數 694 HTTP狀態 {'200': 694}
學歷非空 615 簡歷非空 603


### Sample

In [4]:
display(df[['source_id','expected_name','http_status','name_present','education_raw']].head(10))

,source_id,expected_name,http_status,name_present,education_raw
0,ER11112AA00008,鍾佩玲,200,True,國立政治大學經營管理碩士\r\n世新大學口語傳播學系
1,ER11112AA00002,陳賢蔚,200,True,國立臺灣海洋大學工學學士\r\n臺北市立復興高級中學
2,ER11112AA00005,林世宗,200,True,
3,ER11112AA00010,黃瀞瑩,200,True,倫敦大學金匠學院政治傳播碩士\r\n天主教輔仁大學新聞傳播學系學士\r\n臺北市立明倫高中\...
4,ER11112AA00007,陳慈慧,200,True,開南大學應用日語學系\r\n泰北高中\r\n士林區福安國中\r\n士林區社子國小
5,ER11112AA00006,陳重文,200,True,國立體育大學體研所博士\r\n國立體育大學體育推廣學系碩士
6,ER11112AA00003,林杏兒,200,True,銘傳大學公共事務學系碩士\r\n福安國中\r\n社子國小
7,ER11112AA00011,張斯綱,200,True,國立政治大學東亞所碩士\r\n輔仁大學法律學系財經組
8,ER11112AA00004,汪志冰,200,True,美國東密西根大學文學碩士\r\n東吳大學德文系\r\n文藻外語學院德文科
9,ER11112AA00001,林延鳳,200,True,臺灣大學生物產業傳播暨發展學系碩士


### Summary Stats

In [5]:
display(df.groupby(['county_city','http_status']).size().rename('N').to_frame())
for r in records:
 if r['path']:assert sha(ROOT/r['path'])==r['sha256']
assert df.source_id.is_unique
print('逐檔hash、source_id唯一鍵核對通過；原始資料未變動')

,,N
county_city,http_status,
,200,2
南投縣,200,24
嘉義市,200,18
嘉義縣,200,29
基隆市,200,23
宜蘭縣,200,23
屏東縣,200,38
彰化縣,200,40
新北市,200,48


逐檔hash、source_id唯一鍵核對通過；原始資料未變動
